# Metadata Extraction — Schema v1.3

Extract Pydantic-validated metadata from one production batch and append one record per attempt to the shared JSONL outputs. Run the selection cells before the live-call cell.

For each batch, change only `SNAPSHOTS_DIR`. Successful paths already present in `results.jsonl` are excluded before selection, so completed snapshots remain skipped across batch runs. `EXACT_FILE_PATHS` remains available for targeted diagnosis and takes precedence over `SOURCES`, `TYPES`, and `n_samples`.

In [1]:
%load_ext autotime

import json
import random
import time
from pathlib import Path

from tqdm.auto import tqdm

from data_snapshot.constants import ROOT
from data_snapshot.metadata_extraction import extract_metadata

## Inputs

In [2]:
tag = "batch1"

SOURCES = ["unhcr", "prwp", "refugee"]
TYPES = ["figure", "table"]
n_samples = None  # Process every eligible snapshot in the selected batch.

# Absolute paths or paths relative to ROOT. Nonempty means exact-path mode.
EXACT_FILE_PATHS = []

SLEEP_SECONDS = 0.2
NOTEBOOK_DIR = ROOT / "notebooks/metadata_extraction"
SNAPSHOTS_DIR = NOTEBOOK_DIR / f"data/{tag}"
CONFIG_PATH = ROOT / "src/data_snapshot/metadata_extraction/config/default.json"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
RESULTS_PATH = OUTPUT_DIR / f"{tag}_results.jsonl"
ERRORS_PATH = OUTPUT_DIR / f"{tag}_errors.jsonl"

## Selection helpers

In [3]:
def snapshot_key(path: Path) -> str:
    """Return a stable source/type/filename key for a snapshot path."""
    try:
        relative = path.resolve().relative_to(SNAPSHOTS_DIR.resolve())
    except ValueError as exc:
        raise ValueError(f"Snapshot is outside {SNAPSHOTS_DIR}: {path}") from exc
    if len(relative.parts) != 3 or relative.parts[1] not in {"figure", "table"}:
        raise ValueError(f"Unexpected snapshot layout: {relative}")
    return relative.as_posix()


def resolve_exact_path(value: str | Path) -> Path:
    """Resolve and validate one absolute or repository-relative PNG path."""
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = ROOT / path
    path = path.resolve()
    if not path.is_file() or path.suffix.lower() != ".png":
        raise ValueError(f"Exact snapshot path is not a PNG file: {path}")
    snapshot_key(path)
    return path


def load_completed_paths(path: Path) -> set[str]:
    """Read successful snapshot keys from an existing results JSONL."""
    if not path.exists():
        return set()
    completed = set()
    with path.open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on {path}:{line_number}") from exc
            key = record.get("snapshot_path")
            if not isinstance(key, str) or not key:
                raise ValueError(f"Missing snapshot_path on {path}:{line_number}")
            completed.add(key)
    return completed


def append_jsonl(path: Path, record: dict[str, object]) -> None:
    """Append one JSON-compatible record, creating its directory."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

## Build the skip list before selection

In [4]:
completed_snapshot_paths = load_completed_paths(RESULTS_PATH)
print(f"Completed snapshots in {RESULTS_PATH.name}: {len(completed_snapshot_paths)}")

Completed snapshots in batch1_results.jsonl: 0


## Select snapshots

In [5]:
if not SNAPSHOTS_DIR.is_dir():
    raise ValueError(f"Snapshot directory does not exist: {SNAPSHOTS_DIR}")

selected_snapshots = []

if EXACT_FILE_PATHS:
    seen = set()
    for value in EXACT_FILE_PATHS:
        path = resolve_exact_path(value)
        key = snapshot_key(path)
        if key in completed_snapshot_paths:
            print(f"Already completed: {key}")
        elif key not in seen:
            selected_snapshots.append(path)
            seen.add(key)
    selection_mode = "exact paths"
else:
    if n_samples is not None and (not isinstance(n_samples, int) or n_samples < 0):
        raise ValueError("n_samples must be a non-negative integer or None.")
    for source in SOURCES:
        for artifact_type in TYPES:
            directory = SNAPSHOTS_DIR / source / artifact_type
            eligible = [
                path
                for path in sorted(directory.glob("*.png"))
                if snapshot_key(path) not in completed_snapshot_paths
            ]
            count = (
                len(eligible) if n_samples is None else min(n_samples, len(eligible))
            )
            sampled = eligible if n_samples is None else random.sample(eligible, count)
            selected_snapshots.extend(sampled)
            print(
                f"{source}/{artifact_type}: {len(eligible)} eligible, "
                f"{len(sampled)} selected"
            )
    selected_snapshots.sort(key=snapshot_key)
    selection_mode = (
        "all eligible snapshots" if n_samples is None else "stratified sample"
    )

print(f"Selection mode: {selection_mode}")
print(f"Total selected: {len(selected_snapshots)}")
for path in selected_snapshots:
    print(snapshot_key(path))

unhcr/figure: 4 eligible, 4 selected
unhcr/table: 4 eligible, 4 selected
prwp/figure: 4 eligible, 4 selected
prwp/table: 4 eligible, 4 selected
refugee/figure: 4 eligible, 4 selected
refugee/table: 4 eligible, 4 selected
Selection mode: all eligible snapshots
Total selected: 24
prwp/figure/document_11958451_figure_004.png
prwp/figure/document_12916918_figure_008.png
prwp/figure/document_14861148_figure_004.png
prwp/figure/document_6731296_figure_002.png
prwp/table/document_11332543_table_009.png
prwp/table/document_14861148_table_002.png
prwp/table/document_437268_table_005.png
prwp/table/document_7581269_table_006.png
refugee/figure/121_PAD1190-PAD-P152848-PUBLIC-Box391435B-LB-EESSP-Final-PAD-for-printing_figure_002.png
refugee/figure/189_multi-page_figure_001.png
refugee/figure/196_multi-page_figure_001.png
refugee/figure/197_multi-page_figure_000.png
refugee/table/004_BOSIB-87c444de-4797-4bf9-b654-4932a7fb0112_table_007.png
refugee/table/060_Yemen-Emergency-COVID-19-Project_table_00

## Live API calls

Running the next cell incurs API usage. Successful records go to `RESULTS_PATH`; failures remain retryable and go to `ERRORS_PATH`.

In [6]:
succeeded = failed = skipped = 0

for snapshot_path in tqdm(selected_snapshots, unit="snapshot"):
    key = snapshot_key(snapshot_path)
    if key in completed_snapshot_paths:
        skipped += 1
        continue

    source, artifact_type, _ = key.split("/", maxsplit=2)
    result = extract_metadata(snapshot_path, config_path=CONFIG_PATH)
    record = {
        "snapshot_path": key,
        "snapshot_file_name": snapshot_path.name,
        "source": source,
        "artifact_type": artifact_type,
        "schema_version": "1.3",
        "model": result.model,
        "response_id": result.response_id,
        "api_status": result.api_status,
        "elapsed_seconds": result.elapsed_seconds,
        "usage": result.usage,
    }
    if result.metadata is not None:
        append_jsonl(
            RESULTS_PATH,
            {
                **record,
                "metadata": result.metadata.model_dump(mode="json", exclude_none=True),
            },
        )
        completed_snapshot_paths.add(key)
        succeeded += 1
    else:
        append_jsonl(
            ERRORS_PATH,
            {
                **record,
                "raw_output": result.raw_output,
                "error_type": result.error_type,
                "error": result.error,
            },
        )
        failed += 1
        print(f"{key}: {result.error}")

    if result.elapsed_seconds is not None and SLEEP_SECONDS:
        time.sleep(SLEEP_SECONDS)

  0%|          | 0/24 [00:00<?, ?snapshot/s]

In [7]:
print(f"Succeeded: {succeeded}")
print(f"Failed: {failed}")
print(f"Skipped after selection: {skipped}")
print(f"Results: {RESULTS_PATH}")
print(f"Errors: {ERRORS_PATH}")

Succeeded: 24
Failed: 0
Skipped after selection: 0
Results: /home/ajd/data-snapshot-annotation/notebooks/metadata_extraction/outputs/batch1_results.jsonl
Errors: /home/ajd/data-snapshot-annotation/notebooks/metadata_extraction/outputs/batch1_errors.jsonl
